In [ ]:
import torch


import torch

x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
# tensor([2., 5.])  # 对每一行求平均
print(x.shape)
y = x.mean(dim=1)
print(y)
print(y.shape)

torch.Size([2, 3])
tensor([2., 5.])
torch.Size([2])


In [ ]:
import json

import numpy as np

# 确定相邻span建边的时间
with open("/home/tyf/Project/Tantic/demo.json", "r") as f:
    data = json.load(f)
    start_times = []
    # 是一个 map[map]， 解析所有值
    for key, value in data.items():
        start_time = value["flow_start_time"]
        start_times.append(start_time)
    np_start_times = np.array(start_times)
    np.diff(np_start_times)
    print("mean diff:", np.diff(np_start_times).mean())
    print("std diff:", np.diff(np_start_times).std())
    print("max diff:", np.diff(np_start_times).max())
    print("min diff:", np.diff(np_start_times).min())
    for p in [50, 75, 90, 95, 99]:
        print(f"{p} percentile:", np.percentile(np.diff(np_start_times), p))

mean diff: 0.9766653565799489
std diff: 4.994220452672196
max diff: 29.647451877593994
min diff: 0.0003600120544433594
50 percentile: 0.03272843360900879
75 percentile: 0.15423035621643066
90 percentile: 0.29720602035522453
95 percentile: 0.7482526302337645
99 percentile: 20.127054584026386


In [13]:
import numpy as np

path = "/home/tyf/Project/Tantic/raw_feature/stgc_sp_all_class/X_000.npy"
X = np.load(path)  # shape (num_samples, num_features, num_timesteps)
print("Shape of X:", X.shape)

Shape of X: (10759, 32, 26)


In [ ]:
import os

from arrow import get

# 检查 CIDR 覆盖率
# total_ips: 131099 total_cidrs: 102162
# ratio cidrs/ips: 0.7792736786703178
sample_file_dir = "/home/tyf/fnnas/Study/Traffic-data/train_raw_data"
website_idx = 0


def get_cidr(ip: str) -> str:
    parts = ip.split(".")
    cidr = ".".join(parts[0:3]) + ".0/24"
    return cidr


total_ips = 0
total_cidrs = 0
for website_name in os.listdir(sample_file_dir):
    website_idx = website_idx + 1
    website_folder = os.path.join(sample_file_dir, website_name)
    if not os.path.isdir(website_folder):
        continue

    for instance_id in os.listdir(website_folder):
        data_dir = os.path.join(website_folder, instance_id)
        if not os.path.isdir(data_dir):
            continue
        # Find pcap file and summary file
        pcap_file: str
        summary_file: str

        for filename in os.listdir(data_dir):
            if filename == "summary.txt":
                summary_file = os.path.join(data_dir, filename)

        with open(summary_file, "r") as f:
            lines = f.readlines()
            ips = set()
            cidrs = set()
            # ('10.161.34.27', 58023, '121.194.11.75', 443)
            for line in lines:
                parts = line.strip().split()
                src_ip = parts[0]
                src_port = parts[1]
                dst_ip = parts[2]
                dst_port = parts[3]
                ips.add(dst_ip)
                cidrs.add(get_cidr(dst_ip))
            print("len(ips):", len(ips), "len(cidrs):", len(cidrs))
            total_ips += len(ips)
            total_cidrs += len(cidrs)

print("total_ips:", total_ips, "total_cidrs:", total_cidrs)
print(f"ratio cidrs/ips: {total_cidrs/total_ips}")

len(ips): 23 len(cidrs): 23
len(ips): 15 len(cidrs): 15
len(ips): 14 len(cidrs): 14
len(ips): 15 len(cidrs): 14
len(ips): 16 len(cidrs): 15
len(ips): 14 len(cidrs): 14
len(ips): 19 len(cidrs): 19
len(ips): 16 len(cidrs): 15
len(ips): 20 len(cidrs): 20
len(ips): 15 len(cidrs): 14
len(ips): 16 len(cidrs): 15
len(ips): 13 len(cidrs): 13
len(ips): 24 len(cidrs): 23
len(ips): 20 len(cidrs): 19
len(ips): 23 len(cidrs): 22
len(ips): 16 len(cidrs): 15
len(ips): 13 len(cidrs): 13
len(ips): 15 len(cidrs): 15
len(ips): 17 len(cidrs): 15
len(ips): 15 len(cidrs): 15
len(ips): 14 len(cidrs): 14
len(ips): 16 len(cidrs): 15
len(ips): 17 len(cidrs): 16
len(ips): 21 len(cidrs): 21
len(ips): 23 len(cidrs): 22
len(ips): 22 len(cidrs): 20
len(ips): 1 len(cidrs): 1
len(ips): 17 len(cidrs): 16
len(ips): 15 len(cidrs): 15
len(ips): 12 len(cidrs): 12
len(ips): 14 len(cidrs): 14
len(ips): 16 len(cidrs): 15
len(ips): 15 len(cidrs): 14
len(ips): 20 len(cidrs): 18
len(ips): 18 len(cidrs): 16
len(ips): 21 len(cidrs

In [21]:
# 统计数据集信息
import os
import numpy as np

website_label_name_map = {}
website_idx = 0
sample_file_dir = "/home/tyf/fnnas/Study/Traffic-data/train_raw_data"
for website_name in os.listdir(sample_file_dir):
    website_label_name_map[website_idx] = website_name
    website_idx += 1

print("Website label name map:", website_label_name_map)

data_dir = "/home/tyf/Project/Tantic/raw_feature/stgc_sp_all_class_tls_3"
num_samples = 0
num_classes = set()
class_dict = {}
class_flow_count = {}
for filename in os.listdir(data_dir):
    if filename.endswith(".npy") and filename.startswith("X_"):
        sample_id = filename[2:-4]
        label_file = f"y_{sample_id}.npy"
        if not os.path.exists(os.path.join(data_dir, label_file)):
            continue
        X = np.load(os.path.join(data_dir, filename))
        y = np.load(os.path.join(data_dir, label_file))
        num_samples += X.shape[0]
        for idx in range(y.shape[0]):
            label = int(y[idx])
            num_classes.add(label)
            if label not in class_dict:
                class_dict[label] = 0
            class_dict[label] += 1
            # 计算 flow 数量
            flow_num = 0
            for flow in X[idx]:
                if np.sum(flow) != 0:
                    flow_num += 1
            # print(f"Sample {sample_id} idx {idx} label {label} has {flow_num} flows")
            class_flow_count[label] = class_flow_count.get(label, 0) + flow_num
print("Total samples:", num_samples)
print("Total classes:", len(num_classes))
print("Class distribution:", class_dict)
print("Class flow number distribution:", class_flow_count)
# print website names, instance number, flow_number , csv
print(f"{'label':^12}{'website_name':^20}{'instance_num':^12}{'flow_num':^12}")
for label, name in website_label_name_map.items():
    instance_num = class_dict.get(label, 0)
    flow_num = class_flow_count.get(label, 0)
    # csv line
    print(f"{label:^12}{name:^20}{instance_num:^12}{flow_num:^12}")

Website label name map: {0: 'blog.csdn', 1: 'xiaohongshu.com', 2: 'bilibili.com', 3: 'zhihu.com', 4: 'douban.com', 5: 'ifeng.com', 6: 'v.qq.com', 7: 'jd.com', 8: 'news.cn', 9: 'qq.com', 10: 'souhu.com', 11: 'taobao', 12: 'tieba.baidu.com', 13: 'toutiao.com', 14: 'youku.com', 15: 'vip.com', 16: 'goofish.com'}
Total samples: 10763
Total classes: 17
Class distribution: {5: 126, 13: 1461, 11: 152, 2: 424, 8: 164, 10: 725, 14: 646, 6: 1634, 12: 1690, 0: 640, 1: 638, 9: 955, 3: 446, 4: 444, 16: 190, 7: 294, 15: 134}
Class flow number distribution: {5: 2748, 13: 13992, 11: 1919, 2: 4683, 8: 1538, 10: 8438, 14: 5429, 6: 43541, 12: 38555, 0: 14798, 1: 7865, 9: 9488, 3: 2751, 4: 2678, 16: 1797, 7: 3050, 15: 2409}
   label        website_name    instance_num  flow_num  
     0           blog.csdn          640        14798    
     1        xiaohongshu.com       638         7865    
     2          bilibili.com        424         4683    
     3           zhihu.com          446         2751    
  